In [1]:
# Setup experiment in paths_config.py
# Add the path to the configuration file
# Path to paths_config: /scMEDAL_for_scRNAseq/Experiments/HealthyHeart/paths_config.py
# Only use with python 12
import sys
import pandas as pd
sys.path.append("../../../")
from paths_config import run_names_dict, results_path_dict, compare_models_path

import os
import numpy as np

from scMEDAL.utils.compare_results_utils_python12 import (
    aggregate_paths,
    read_and_aggregate_scores,
    filter_min_max_silhouette_scores,
    process_all_results,
    process_confidence_intervals,
)


"""
This script reorganizes clustering scores into a single table. 
It also adds 95% confidence intervals (CI) to the results.
Environment: run_models_env
"""

# --------------------------------------------------------------------------------------
# 1. Define dataset type and output directory
# --------------------------------------------------------------------------------------
# Set the dataset type: Use 'val' for development or 'test' for final results
dataset_type = 'test'

# Define output directory for results
out_name = os.path.join(compare_models_path, run_names_dict["run_name_all"])

# Ensure the directory exists
if not os.path.exists(out_name):
    os.makedirs(out_name)
print(f"Directory created or already exists: {out_name}")

# --------------------------------------------------------------------------------------
# 2. Get paths for mean and all scores
# --------------------------------------------------------------------------------------
# Aggregate paths for mean scores
df_all_paths = aggregate_paths(results_path_dict, pattern=f'mean_scores_{dataset_type}_samplesize')
print("\n\nMean scores paths:")
print(df_all_paths.head())

# Aggregate paths for all scores
df_all_paths_allscores = aggregate_paths(results_path_dict, pattern=f'all_scores_{dataset_type}_samplesize')
print("\nAll scores paths:")
print(df_all_paths_allscores)

print("\nColumns in all scores paths:")
print(df_all_paths_allscores.columns)


/tmp/ipykernel_39905/2867246940.py:6: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


data_base_path: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../data/ASD_data
outputs_path: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs


/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, F

Directory created or already exists: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/compare_models/log_transformed_2916hvggenes/DefineGeneralname4yourexpt
scMEDAL-RE /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scMEDAL-RE/scMEDALpaperresults_run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2024-07-28_23-18
scVI /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scVI/run_crossval_n_latent_dims-2_n_layers-2_n_hidden-132_gene_likelihood-zinb_dispersion-gene_scaling-min_max_batch_size-512_epochs-2_patience-30_compute_latents_callback-False_samp

In [2]:
def get_named_columns(df: pd.DataFrame) -> pd.MultiIndex:
    """
    Returns a MultiIndex of columns that do not contain 'Unnamed' in any level.
    """
    return df.columns[
        ~df.columns.to_frame(index=False).apply(lambda col: col.str.contains("Unnamed"), axis=1).any(axis=1)
    ]

def read_and_aggregate_scores(df_all_paths_allscores):
    """
    Reads score CSVs, keeps only named columns, and adds 'fold', 'index', 'sample_size', and 'latent_name'.
    """
    df_list = []

    for _, row in df_all_paths_allscores.iterrows():
        path = row['path']
        sample_size = row['sample_size']
        model_name = row['model_name']

        df = pd.read_csv(path, header=[0, 1], index_col=None)
        # print(df)
        cols = get_named_columns(df)
        df_new = df.loc[:, cols].copy()

        df_new["index"] = df.index
        df_new["fold"] = df["fold"].values.T[0]
        df_new["sample_size"] = sample_size
        df_new["latent_name"] = model_name

        df_list.append(df_new)

    return pd.concat(df_list, ignore_index=True)

In [3]:
df_all_paths_allscores

,sample_size,path,model_name
0,10000,/endosome/archive/bioinformatics/DLLab/src/Aix...,scMEDAL-RE
1,10000,/endosome/archive/bioinformatics/DLLab/src/Aix...,scVI


In [6]:
df_all_paths_allscores.loc[1,"path"]


'/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scVI/run_crossval_n_latent_dims-2_n_layers-2_n_hidden-132_gene_likelihood-zinb_dispersion-gene_scaling-min_max_batch_size-512_epochs-2_patience-30_compute_latents_callback-False_sample_size-10000_model_type-ae_n_components-2_2025-05-27_21-46/all_scores_test_samplesize-10000.csv'

In [6]:
# --------------------------------------------------------------------------------------
# 3. Read all scores and save to a CSV file
# --------------------------------------------------------------------------------------
df_allscores = read_and_aggregate_scores(df_all_paths_allscores)
print("\nAggregated scores DataFrame:")
print(df_allscores)

# Save all scores to CSV
df_allscores.to_csv(os.path.join(out_name, f"{dataset_type}_allscores.csv"))


Aggregated scores DataFrame:
          batch                          celltype                      index  \
             ch      1/db silhouette           ch      1/db silhouette         
0    436.742535  0.095873  -0.171453   261.978130  0.015703  -0.270995     0   
1    581.441024  0.108810  -0.140995   251.489115  0.034927  -0.248158     1   
2     67.917069  0.037736  -0.413081  1297.475035  0.021829  -0.192891     2   
3     69.231420  0.034046  -0.223528  1908.955905  0.041987  -0.088378     3   
4  10964.603768  0.500065   0.405638    27.129653  0.014496  -0.158317     4   
5     15.789402  0.024478  -0.177803  7961.576645  0.463184   0.341057     0   
6     11.993127  0.017962  -0.177296  7762.190689  0.373772   0.306177     1   
7     11.626820  0.022950  -0.179101  8631.403896  0.330516   0.350104     2   
8     13.528066  0.024128  -0.212296  8579.386810  0.337105   0.323040     3   
9     14.321249  0.015897  -0.200440  8617.400530  0.394302   0.326058     4   

  fold sa

In [17]:

# --------------------------------------------------------------------------------------
# 4. Filter minimum and maximum silhouette scores for the batch
# --------------------------------------------------------------------------------------
# for sample_size in np.unique(df_allscores["sample_size"]):
#     # Filter minimum and maximum silhouette scores for the "batch" column
#     df_min_silhouette, df_max_silhouette = filter_min_max_silhouette_scores(df_allscores, batch_col="batch")
    
#     # Save the filtered results to CSV
#     df_min_silhouette.to_csv(os.path.join(out_name, f"{dataset_type}_scores_{sample_size}_min_silhouette_batch.csv"))
#     df_max_silhouette.to_csv(os.path.join(out_name, f"{dataset_type}_scores_{sample_size}_max_silhouette_batch.csv"))

#     # Print the resulting DataFrames
#     print("DataFrame with minimum silhouette scores:")
#     print(df_min_silhouette)

#     print("\nDataFrame with maximum silhouette scores:")
#     print(df_max_silhouette)

# --------------------------------------------------------------------------------------
# 5. Process results depending on the model configuration
# --------------------------------------------------------------------------------------
# Define how to process results for each model
# If `get_pca=True` for any model in the pipeline, use "preprocess_results_model_pca_format".
# Otherwise, use "process_single_model_format" or leave empty if no processing is required.
# NOTE: Keys in models2process_dict must match those in run_names_dict.
models2process_dict = {
    # "AE": "preprocess_results_model_pca_format",
    # "AEC": "preprocess_results_model_pca_format",
    # "scMEDAL-FEC": "process_single_model_format",
    # "scMEDAL-FE": "process_single_model_format",
    "scMEDAL-RE": "process_single_model_format",
    "scVI": "process_single_model_format"
}

# Process all results
df_sample_size = process_all_results(df_all_paths, models2process_dict, out_name, dataset_type)

# --------------------------------------------------------------------------------------
# 6. Calculate 95% confidence intervals (CI) for results
# --------------------------------------------------------------------------------------
for sample_size, df_all in df_sample_size.items():
    df_mean_ci_results = process_confidence_intervals(df_all, out_name, dataset_type, sample_size)
    print(f"Sample size: {sample_size}\nConfidence interval results:")
    print(df_mean_ci_results)


Sample size: 10000
scMEDAL-RE file path: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scMEDAL-RE/scMEDALpaperresults_run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2024-07-28_23-18/mean_scores_test_samplesize-10000.csv
reading file: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scMEDAL-RE/scMEDALpaperresults_run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2024-07-28_23-18/mean_scores_test_samplesize-10000.csv
scVI file path: /endosome/archive/bioinformatics/DLLab/src/A

In [8]:
df_all

batch                                                \
                       ch                                1/db             
                     mean          std          sem      mean       std   
dataset_type                                                              
scMEDAL-RE    2423.987163  4779.701622  2137.547548  0.155306  0.195627   
scVI            22.450771     3.653122     1.633726  0.022348  0.004938   

                                                           celltype  \
                       silhouette                                ch   
                   sem       mean       std       sem          mean   
dataset_type                                                          
scMEDAL-RE    0.087487  -0.108684  0.306332  0.136996    749.405568   
scVI          0.002208  -0.247962  0.032862  0.014696  10198.050038   

                                                                      \
                                            1/db                       
                      std          sem      mean       std       sem   
dataset_type                                                           
scMEDAL-RE     814.267450   364.151474  0.025788  0.012153  0.005435   
scVI          2541.140177  1136.432435  0.221406  0.096660  0.043227   

                                             
             silhouette                      
                   mean       std       sem  
dataset_type                                 
scMEDAL-RE    -0.191748  0.072915  0.032609  
scVI           0.169994  0.040578  0.018147

In [9]:
df_sample_size["10000"]

batch                                                \
                       ch                                1/db             
                     mean          std          sem      mean       std   
dataset_type                                                              
scMEDAL-RE    2423.987163  4779.701622  2137.547548  0.155306  0.195627   
scVI            22.450771     3.653122     1.633726  0.022348  0.004938   

                                                           celltype  \
                       silhouette                                ch   
                   sem       mean       std       sem          mean   
dataset_type                                                          
scMEDAL-RE    0.087487  -0.108684  0.306332  0.136996    749.405568   
scVI          0.002208  -0.247962  0.032862  0.014696  10198.050038   

                                                                      \
                                            1/db                       
                      std          sem      mean       std       sem   
dataset_type                                                           
scMEDAL-RE     814.267450   364.151474  0.025788  0.012153  0.005435   
scVI          2541.140177  1136.432435  0.221406  0.096660  0.043227   

                                             
             silhouette                      
                   mean       std       sem  
dataset_type                                 
scMEDAL-RE    -0.191748  0.072915  0.032609  
scVI           0.169994  0.040578  0.018147

In [10]:
df_all

batch                                                \
                       ch                                1/db             
                     mean          std          sem      mean       std   
dataset_type                                                              
scMEDAL-RE    2423.987163  4779.701622  2137.547548  0.155306  0.195627   
scVI            22.450771     3.653122     1.633726  0.022348  0.004938   

                                                           celltype  \
                       silhouette                                ch   
                   sem       mean       std       sem          mean   
dataset_type                                                          
scMEDAL-RE    0.087487  -0.108684  0.306332  0.136996    749.405568   
scVI          0.002208  -0.247962  0.032862  0.014696  10198.050038   

                                                                      \
                                            1/db                       
                      std          sem      mean       std       sem   
dataset_type                                                           
scMEDAL-RE     814.267450   364.151474  0.025788  0.012153  0.005435   
scVI          2541.140177  1136.432435  0.221406  0.096660  0.043227   

                                             
             silhouette                      
                   mean       std       sem  
dataset_type                                 
scMEDAL-RE    -0.191748  0.072915  0.032609  
scVI           0.169994  0.040578  0.018147

In [11]:
from scMEDAL.utils.compare_results_utils_python12 import process_single_model_format
process_single_model_format(file_path= df_all_paths_allscores['path'][0], model_name="scMEDAL-RE")

reading file: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scMEDAL-RE/scMEDALpaperresults_run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2024-07-28_23-18/all_scores_test_samplesize-10000.csv


NaN                                                   \
                  ch                                                    
             batch.1     batch.2 celltype celltype.1  celltype.2 fold   
dataset_type                                                            
scMEDAL-RE      1/db  silhouette       ch       1/db  silhouette  NaN   

                                           0.0                        \
                             436.7425352128527                         
             dataset_type              batch.1               batch.2   
dataset_type                                                           
scMEDAL-RE            NaN  0.09587284694086184  -0.17145341634750366   

                                 ...                  3.0       \
                                 ...    69.23142019953092        
                       celltype  ...           celltype.2 fold   
dataset_type                     ...                             
scMEDAL-RE    261.9781297685927  ...  -0.0883776843547821  4.0   

                                                  4.0                      \
                                    10964.60376759362                       
                     dataset_type             batch.1             batch.2   
dataset_type                                                                
scMEDAL-RE    RE_AE_latent_2_test  0.5000648999037585  0.4056376814842224   

                                                                              \
                                                                               
                        celltype            celltype.1            celltype.2   
dataset_type                                                                   
scMEDAL-RE    27.129652842756677  0.014495737118861087  -0.15831725299358368   

                                        
                                        
             fold         dataset_type  
dataset_type                            
scMEDAL-RE    5.0  RE_AE_latent_2_test  

[1 rows x 42 columns]

In [16]:
df_out = process_single_model_format(file_path= path, model_name="scVI")

reading file: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scVI/run_crossval_n_latent_dims-2_n_layers-2_n_hidden-132_gene_likelihood-zinb_dispersion-gene_scaling-min_max_batch_size-512_epochs-2_patience-30_compute_latents_callback-False_sample_size-10000_model_type-ae_n_components-2_2025-05-27_21-46/mean_scores_test_samplesize-10000.csv


In [14]:
path= "/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scVI/run_crossval_n_latent_dims-2_n_layers-2_n_hidden-132_gene_likelihood-zinb_dispersion-gene_scaling-min_max_batch_size-512_epochs-2_patience-30_compute_latents_callback-False_sample_size-10000_model_type-ae_n_components-2_2025-05-27_21-46/mean_scores_test_samplesize-10000.csv"

In [ ]:
def process_single_model_format(file_path, model_name):
    """
    Process a CSV with (index=[0,1]) and multi-index columns, dropping 'fold' if present at any level.
    """
    print("reading file:", file_path)

    # Load DataFrame (MultiIndex on index, single-level columns)
    df = pd.read_csv(file_path, header=[0], index_col=[0, 1]).T

    # Drop any columns with 'fold' in any level of the column tuple
    df = df[[col for col in df.columns if 'fold' not in col]]

    # Convert to dict for flattening
    data = df.to_dict()

    # Build MultiIndex columns
    columns = pd.MultiIndex.from_tuples([(key[0], key[1], stat) for key in data for stat in data[key]])
    values = [data[key][stat] for key in data for stat in data[key]]

    # Recreate DataFrame
    new_df = pd.DataFrame([values], columns=columns)
    new_df["dataset_type"] = model_name
    new_df.set_index("dataset_type", inplace=True)

    return new_df

In [12]:
A= process_single_model_format(file_path= df_all_paths_allscores['path'][1], model_name="scVI")

reading file: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/scVI/run_crossval_n_latent_dims-2_n_layers-2_n_hidden-132_gene_likelihood-zinb_dispersion-gene_scaling-min_max_batch_size-512_epochs-2_patience-30_compute_latents_callback-False_sample_size-10000_model_type-ae_n_components-2_2025-05-27_21-46/all_scores_test_samplesize-10000.csv


In [13]:
A

ch                                                    \
                    1/db                                                     
                 batch.2 celltype celltype.1  celltype.2 fold dataset_type   
dataset_type                                                                 
scVI          silhouette       ch       1/db  silhouette  NaN          NaN   

                 26.26507361293684                                        \
              0.022152748818677764                                         
                           batch.2            celltype        celltype.1   
dataset_type                                                               
scVI          -0.19440706074237823  10431.808765038613  0.32665243805043   

                                  ...   18.800816105643253  \
                                  ... 0.024806569828866608   
                      celltype.2  ...           celltype.1   
dataset_type                      ...                        
scVI          0.2236541211605072  ...  0.07872959928159179   

                                                            \
                                                             
                       celltype.2 fold        dataset_type   
dataset_type                                                 
scVI          0.11477973312139511  4.0  scVI_latent_2_test   

                19.64711328651388                                        \
             0.025656617796598495                                         
                          batch.2          celltype          celltype.1   
dataset_type                                                              
scVI          -0.2683887481689453  12232.7587300189  0.2941912097573952   

                                                            
                                                            
                       celltype.2 fold        dataset_type  
dataset_type                                                
scVI          0.18383535742759705  5.0  scVI_latent_2_test  

[1 rows x 36 columns]

In [16]:
df_all_paths_allscores

,sample_size,path,model_name
0,10000,/endosome/archive/bioinformatics/DLLab/src/Aix...,scMEDAL-RE
1,10000,/endosome/archive/bioinformatics/DLLab/src/Aix...,scVI


In [ ]:
        df_smf_list.append(process_single_model_format(file_path=file_path, model_name=model_name))
            

In [ ]:

def process_all_results(df_all_paths, models2process_dict, out_name, dataset_type):
    """
    Process and merge results from different models and sample sizes into a single DataFrame, and save to CSV.

    Parameters:
    df_all_paths (pd.DataFrame): DataFrame with columns 'sample_size', 'model_name', and 'path'.
    models2process_dict (dict): Dict indicating the processing type for each model.
    out_name (str): Directory to save processed CSVs.
    dataset_type (str): Label for the dataset type being processed.

    Returns:
    dict: sample_size ? DataFrame mapping.
    """
    df_sample_size = {}

    for sample_size in np.unique(df_all_paths["sample_size"]):
        print(f"\nSample size: {sample_size}")
        df_smf_list = []
        df_pcaf = pd.DataFrame()
        count_i = 0

        for model_name in np.unique(df_all_paths["model_name"]):
            file_paths = df_all_paths.loc[
                (df_all_paths["sample_size"] == sample_size) & 
                (df_all_paths["model_name"] == model_name), 
                "path"
            ].values

            if len(file_paths) == 0:
                print(f"{model_name} has no path for sample size {sample_size}")
                continue

            file_path = file_paths[0]
            print(f"{model_name} file path: {file_path}")

            process_type = models2process_dict.get(model_name, None)

            if process_type == "process_single_model_format":
                df_smf_list.append(process_single_model_format(file_path=file_path, model_name=model_name))
            
            elif process_type == "preprocess_results_model_pca_format":
                df_pcaf_i = preprocess_results_model_pca_format(file_path, columns_to_drop=['fold', 'sem_fold'])
                df_pcaf_i = df_pcaf_i.rename(columns={"X_pca_val": f"X_pca_val_{count_i}"})
                df_pcaf = pd.concat([df_pcaf, df_pcaf_i], axis=1)
                count_i += 1

            else:
                print(f"{model_name} not processed (unknown format)")

        df_smf = pd.concat(df_smf_list).T if df_smf_list else pd.DataFrame()

        # Combine both formats
        df_all = pd.concat([df_pcaf, df_smf], axis=1).T
        output_path = os.path.join(out_name, f"{dataset_type}_scores_{sample_size}.csv")
        df_all.to_csv(output_path)
        print(f"Saved: {output_path}")
        df_sample_size[sample_size] = df_all

    return df_sample_size



